# 18 — Roadmap item 5: fixed-epoch training on 100% of the outer-train fold (no early stopping)

**Why** (`project_dat_parkinson_strategic_roadmap.md`, item 5): every rung-3
and rung-4 training run picks its checkpoint via early stopping on a single
small (~109-136 row) inner-validation split (`evaluate.make_folds(...,
n_splits=10)[0]`, never repeated) -- the suspected dominant source of the
~0.007-0.021 per-run sd that's been swamping small-to-moderate real effects
all through rung 4. Opus's recommended fix isn't "average more inner
splits" (that only reduces variance, and notebook 16's ensembling already
captures most of that benefit) -- it's to **remove the inner split
entirely**, train on the full outer-train fold, and use a **fixed epoch
budget** instead of a noisy stopping decision. This should make each member
genuinely better (more training data, no premature stopping) rather than
just less variable, so it compounds with ensembling instead of duplicating
it.

**Fixed epoch budget, derived from real data, not guessed**: extracted
every printed per-fold epoch count from the completed nested-CV runs
(notebooks 07, 09, 10, 12, 13 -- 128 fold-trainings total): median=21,
mean=22.3, range 12-38. This notebook uses **`FIXED_EPOCHS=22`**.

**No inner validation split in this experiment, on purpose**: `train_loader`
is passed as both the train and "val" argument to `train_one_fold` (patience
set to `FIXED_EPOCHS + 1` so early stopping structurally cannot fire) --
there is no honest held-out signal within a fold to select a "best epoch"
from, and that's the point: the outer fold itself is the only held-out data
that matters here, and `best_state` just ends up being (very close to) the
final epoch's weights.

**Gating discipline (Opus's explicit instruction)**: gate this on the
**ensemble metric**, not a per-repeat mean, and as **one single
pre-registered comparison** -- does adding this as a 6th variant to
notebook 16's already-adopted 5-variant ensemble (log loss 0.4022, the
current production composition) improve the honest ensemble OOF? No subset
search.

**Cost**: GPU required (this is a [RUN ME] step touching real `.nii.gz`
volumes and row-level labels, same data rule as notebooks 09-13). Expect
somewhat longer than a typical rung-4 notebook -- fixed 22 epochs (vs. up
to 50 with early stopping) but every epoch does a full pass over 100% of
the outer-train fold twice (train + the same-size pseudo-"val" pass),
instead of training on ~72% with a small ~8%-of-outer validation slice.
Order of magnitude: still well under an hour for the full 5x5.

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels. Reuses the shared
# on-disk volume cache (this experiment doesn't touch preprocessing).
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

In [ ]:
# [RUN ME] (no data access itself). Trains on the FULL outer-train fold --
# no inner split. `train_loader` doubles as the "val" loader purely so
# train_one_fold's existing (tested) API can be reused; patience is set
# above FIXED_EPOCHS so it structurally cannot trigger early stopping.
FIXED_EPOCHS = 22  # median=21, mean=22.3 across 128 real fold-trainings (notebooks 07/09/10/12/13)
batch_size, lr = 32, 2e-3  # rung 2/3's validated winner, held fixed here


def train_and_score_full_fold(train_uids, train_labels, outer_uids, seed):
    train_ds = dataset.DatParkinsonDataset(train_uids, train_labels, load_fn=volume_cache.get)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, train_loader, train_loader, optimizer, loss_fn,
        epochs=FIXED_EPOCHS, patience=FIXED_EPOCHS + 1, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [ ]:
# [RUN ME] -- full 5-fold nested CV (no inner split), repeated 5x. Same
# outer-fold protocol as rung 3 (notebooks/07_cnn_rung3.ipynb). Prints
# per-fold timing + a running ETA (25 folds total: 5 seeds x 5 folds) so
# progress is visible without guessing.
N_REPEATS = 5
TOTAL_FOLDS = N_REPEATS * config.N_FOLDS
oof_repeats_fixedepoch = []
fold_durations = []
notebook_start = time.time()

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        fold_start = time.time()
        probs, history, best_state = train_and_score_full_fold(
            fold_train_uids, fold_train_labels, fold_test_uids, seed=repeat_seed,
        )
        fold_elapsed = time.time() - fold_start
        fold_durations.append(fold_elapsed)

        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_fixedepoch_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)

        folds_done = len(fold_durations)
        avg_fold_time = np.mean(fold_durations)
        eta_seconds = avg_fold_time * (TOTAL_FOLDS - folds_done)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['train_loss'])} epochs (fixed), "
              f"outer fold log loss={fold_score:.4f}, {fold_elapsed:.1f}s "
              f"({fold_elapsed / len(history['train_loss']):.2f}s/epoch) -- "
              f"{folds_done}/{TOTAL_FOLDS} folds done, "
              f"avg {avg_fold_time:.1f}s/fold, ETA {eta_seconds / 60:.1f} min")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_fixedepoch.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f} "
          f"(elapsed so far: {(time.time() - notebook_start) / 60:.1f} min)")
    np.save(config.DATA_PROCESSED / f"rung4_fixedepoch_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_fixedepoch = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_fixedepoch])
print(f"\n{N_REPEATS}-repeat fixed-epoch (no early stopping) CNN pooled log loss: "
      f"mean={repeat_scores_fixedepoch.mean():.4f}, sd={repeat_scores_fixedepoch.std(ddof=1):.4f}")
print(f"total training time: {(time.time() - notebook_start) / 60:.1f} min")
print("for reference (per-repeat means, not the ensemble metric this experiment is gated on): "
      "rung3=0.4520, familybias/lrsched/augment/classweight logged in README.md's rung-4 section")

In [ ]:
# [RUN ME] (no new data access -- uses arrays from this notebook + disk).
# THE single pre-registered comparison (Opus's explicit instruction): does
# adding fixed-epoch as a 6th variant to notebook 16's already-adopted
# 5-variant ensemble improve the honest ensemble OOF? No subset search.
current_variant_prefixes = ["rung3", "rung4_familybias", "rung4_lrsched", "rung4_augment", "rung4_classweight"]
repeat_seeds = list(range(config.SEED, config.SEED + N_REPEATS))
y_true = np.array(labels)

five_variant_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in current_variant_prefixes for s in repeat_seeds
]
six_variant_arrays = five_variant_arrays + oof_repeats_fixedepoch

five_variant_ensemble_oof = np.mean(five_variant_arrays, axis=0)
six_variant_ensemble_oof = np.mean(six_variant_arrays, axis=0)

five_scores = evaluate.combined_score(y_true, five_variant_ensemble_oof)
six_scores = evaluate.combined_score(y_true, six_variant_ensemble_oof)

print(f"5-variant ensemble (current production, 25 arrays): log loss={five_scores['log_loss']:.4f}  "
      f"AUROC={five_scores['auroc']:.4f}  ECE={five_scores['ece']:.4f}")
print(f"6-variant ensemble (+fixed-epoch, 30 arrays):        log loss={six_scores['log_loss']:.4f}  "
      f"AUROC={six_scores['auroc']:.4f}  ECE={six_scores['ece']:.4f}")
print(f"\ndelta (6-variant - 5-variant): {six_scores['log_loss'] - five_scores['log_loss']:+.4f} log loss")
print("THE DECISION RULE: if this delta is negative, adopt fixed-epoch as a 6th ensemble "
      "member (then re-run notebook 17's calibration+blend re-tuning against the new "
      "6-variant composition). If positive/flat, discard fixed-epoch -- do NOT go looking "
      "for a different subset that happens to include it.")

**What we're looking for:** does removing the noisy early-stopping
selection signal and training on 100% of the outer-train fold for a fixed,
data-derived epoch budget produce a genuinely better (not just
lower-variance) ensemble member -- one that measurably improves the
already-adopted 5-variant ensemble when added as a 6th?

**What we found:** total training time 13.2 min (25 folds, ~31.7s/fold).
Fixed-epoch alone: 5-repeat pooled mean=0.4475, sd=0.0122 -- a middling
performer on its own (between familybias/classweight and lrsched/augment),
not a standout individually. The metric that matters, the pre-registered
ensemble comparison: **5-variant ensemble log loss=0.4022, AUROC=0.8984,
ECE=0.0360** vs. **6-variant (+fixed-epoch) log loss=0.3978, AUROC=0.9004,
ECE=0.0305** -- delta **-0.0044**, and all three metrics move the same
(favorable) direction together (log loss down, AUROC up, ECE down). Smaller
in magnitude than notebook 16's own ensembling win (-0.0105) but coherent,
not noise.

**Decision: ADOPT fixed-epoch as a 6th ensemble member** (per the
pre-registered decision rule: negative delta -> adopt, no subset search).
Production composition is now 6 variants / 150 checkpoints: rung3,
familybias, lrsched, augment, classweight, fixedepoch.

**Next step:** re-run notebook 17's LOFO calibration + logit-space blend
re-tuning against this new 6-variant composition (T_cnn/T_baseline/w need
re-fitting again, same as when notebook 16 changed the composition between
15 and 17) -- see `notebooks/19_calibration_blend_retune_sixvariants.ipynb`.
See `project_dat_parkinson_strategic_roadmap.md` for roadmap items 6/7
still open after that.